# Declare a new run

Bare-minimum template for starting a new run: define its sources, its
tokenizer (reused if one already matches), a dataset and a pretraining config
under a fresh `run_id`, then check and declare it. Copy this notebook per run
and change `RUN_ID` and the knobs below. This only declares up through the
`Pretraining` artifact; it does not run any jobs.

`ROOT` is where everything is declared: the volume's mount inside the lab
container, or a folder of your own on a laptop. Pick the matching line in
the next cell.

In [2]:
from pathlib import Path

import lab
from artifacts.core.resolve import resolve
from config import STORAGE

# ROOT = Path(STORAGE)                        # the volume, inside the lab container
ROOT = Path(".scratch/storage").resolve()  # a folder of your own, on a laptop


In [3]:
from artifacts.core.artifact import Resources
from artifacts.core.SGD.training import (
    LoopConfig,
    LRSchedule,
    OptimizerParameters,
    TrainingParameters,
)
from artifacts.dataset import DataSet
from artifacts.mappeddataset import MappedDataSet
from artifacts.models.transformer import ModelParameters, Pretraining
from artifacts.sources import SourceURL
from artifacts.tokenizers.bpe import Tokenizer as BPETokenizer

odyssey = SourceURL(name="odyssey", url="https://www.gutenberg.org/cache/epub/1727/pg1727.txt")
mobydick = SourceURL(name="mobydick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt")
romeojuliet = SourceURL(name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt")
montecristo = SourceURL(name="montecristo", url="https://www.gutenberg.org/cache/epub/1184/pg1184.txt")
pride = SourceURL(name="pride", url="https://www.gutenberg.org/cache/epub/1342/pg1342.txt")
frankenstein = SourceURL(name="frankenstein", url="https://www.gutenberg.org/cache/epub/84/pg84.txt")
# greatexpectations = SourceURL(name="greatexpectations", url="https://www.gutenberg.org/cache/epub/1400/pg1400.txt")
# dracula = SourceURL(name="dracula", url="https://www.gutenberg.org/cache/epub/345/pg345.txt")
# alice = SourceURL(name="alice", url="https://www.gutenberg.org/cache/epub/11/pg11.txt")
# sherlock = SourceURL(name="sherlock", url="https://www.gutenberg.org/cache/epub/1661/pg1661.txt")
# warandpeace = SourceURL(name="warandpeace", url="https://www.gutenberg.org/cache/epub/2600/pg2600.txt")
# taleoftwocities = SourceURL(name="taleoftwocities", url="https://www.gutenberg.org/cache/epub/98/pg98.txt")
# janeeyre = SourceURL(name="janeeyre", url="https://www.gutenberg.org/cache/epub/1260/pg1260.txt")
# huckfinn = SourceURL(name="huckfinn", url="https://www.gutenberg.org/cache/epub/76/pg76.txt")
# wutheringheights = SourceURL(name="wutheringheights", url="https://www.gutenberg.org/cache/epub/768/pg768.txt")

RUN_ID = "MODEL_LEGS"  # <- change this per run

tokenizer = BPETokenizer(
    vocab_size=1000,
    special_tokens=("<pad>", "<unk>"),
    sources=(odyssey, mobydick),
)


justdataset = DataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=(
        montecristo,
        mobydick,
        frankenstein,
    ),
    valid_sources=(romeojuliet, pride),
)


mappedset = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=(
        montecristo,
        mobydick,
        frankenstein,
    ),
    valid_sources=(romeojuliet, pride),
)


mappedset2 = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=(
        montecristo,
        pride,
        frankenstein,
    ),
    valid_sources=(romeojuliet, pride),
)

optimizer_parameters = OptimizerParameters(
    lr= 0.001,
    betas= (0.9, 0.999),
    weight_decay = 0.1,
    eps = 1e-8,
)

model_parameters = ModelParameters(
    vocab_size=tokenizer.vocab_size,
    sequence_length=64,
    num_layers=1,
    d_model=64,
    d_ff=128,
    num_heads=4,
    rope_theta=10000,
    device="cuda",  # has to agree with app.GPU; the preflight checks it
    dtype="torch.float32", #strings parsed in the worker container.
)

lr_schedule = LRSchedule(
    max_learning_rate=1e-3,
    min_learning_rate=1e-4,
    warmup_iters=100,
    cosine_cycle_iters=10000,
)

training_parameters = TrainingParameters(
    total_steps=1000,
    batch_size=64,
    max_norm=1.0,
    lr_schedule=lr_schedule,
    optimizer="torch.optim.AdamW",
    optimizer_parameters=optimizer_parameters,
    seed=0,
)

loop_config = LoopConfig(
    checkpoint_every=500,
    val_every=5000,
    gpu_check_every=100,
    optimizer_checkpoint_policy="latest",
)

pretraining_0 = Pretraining(
    run_id=RUN_ID,
    dataset=mappedset,
    tokenizer=tokenizer,
    model_parameters=model_parameters,
    training_parameters=training_parameters,
    loop_config=loop_config,
    allocated_resources=Resources(gpu_type="T4", gpu_count=1),
)

pretraining_1 = Pretraining(
    run_id=RUN_ID,
    dataset=mappedset2,
    tokenizer=tokenizer,
    starting_checkpoint=pretraining_0,
    training_parameters=training_parameters,
    loop_config=loop_config,
    allocated_resources=Resources(gpu_type="T4", gpu_count=1),
)

pretraining_2 = Pretraining(
    run_id=RUN_ID,
    dataset=mappedset,
    tokenizer=tokenizer,
    starting_checkpoint=pretraining_1,
    training_parameters=training_parameters,
    loop_config=loop_config,
    allocated_resources=Resources(gpu_type="T4", gpu_count=1),
)

pretraining = pretraining_2

## Declare

`resolve(pretraining)` lists what has to exist to get it, in the order it
has to happen, without reading anything. One job produces one artifact, so
that list *is* the work.

`lab.declare(pretraining, root=ROOT)` compares the request against what is
on disk and prints one row per artifact: everything shared (sources, and the
tokenizer if it matches one already declared) should read `done` or
`declared`; everything new to this run should read `new`. `commit=True`
publishes whatever is `new` and refuses outright if anything blocks. A
`conflict` is a different definition already declared at the same path; an
`undeclared` folder holds files nothing declared. `verbose=True` shows what
differs under each blocking row. Commit drift never blocks unless you ask
with `strict_commit=True`.

In [4]:
for artifact in resolve(justdataset):
    print(f"{type(artifact).__name__:16} {artifact.artifact_path}")

Source           sources/montecristo
Source           sources/mobydick
Source           sources/odyssey
Tokenizer        tokenizers/bpe-1000-426f51973f
TokenizedSource  tokenizers/bpe-1000-426f51973f/bin/montecristo
TokenizedSource  tokenizers/bpe-1000-426f51973f/bin/mobydick
Source           sources/frankenstein
TokenizedSource  tokenizers/bpe-1000-426f51973f/bin/frankenstein
Source           sources/romeojuliet
TokenizedSource  tokenizers/bpe-1000-426f51973f/bin/romeojuliet
Source           sources/pride
TokenizedSource  tokenizers/bpe-1000-426f51973f/bin/pride
DataSet          datasets/dataset-2ffa50681b


In [6]:
report = lab.declare(justdataset, root=ROOT, verbose=True, commit=True)

sources/montecristo                              declared  (created)
sources/mobydick                                 declared  (created)
sources/odyssey                                  declared  (created)
tokenizers/bpe-1000-426f51973f                   declared  (created)
tokenizers/bpe-1000-426f51973f/bin/montecristo   declared  (created)
tokenizers/bpe-1000-426f51973f/bin/mobydick      declared  (created)
sources/frankenstein                             declared  (created)
tokenizers/bpe-1000-426f51973f/bin/frankenstein  declared  (created)
sources/romeojuliet                              declared  (created)
tokenizers/bpe-1000-426f51973f/bin/romeojuliet   declared  (created)
sources/pride                                    declared  (created)
tokenizers/bpe-1000-426f51973f/bin/pride         declared  (created)
datasets/dataset-2ffa50681b                      declared  (created)

13 declared
13 manifests published: sources/montecristo, sources/mobydick, sources/odyssey, tokenizers

In [ ]:
justdataset.bind(ROOT)

TypeError: Artifact.load() missing 1 required positional argument: 'artifact_path'